In [1]:
import sys
import os
sys.path.append(os.path.abspath("../.."))
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, SeasonalNaive
import time
from tinyconformal.series import ConformalQuantileTimeSeriesRegressor
from tinyshift.modelling import DMSTLWrapper, fourier_seasonality
from utilsforecast.preprocessing import fill_gaps
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from mlforecast import MLForecast
from statsforecast.models import SeasonalNaive, AutoETS
import lightgbm as lgb
# Gerando dados em painel sintéticos para 5 séries temporais
np.random.seed(42)
n_series = 5
time_steps = 100

url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv'
df = pd.read_csv(url, parse_dates=['Month'])
df["unique_id"] = "1"
df.rename(columns={"Month": "ds", "Passengers": "y"}, inplace=True)
df = fill_gaps(df, freq="ME", end="per_serie", id_col="unique_id", time_col="ds")
df = fourier_seasonality(df, "ds", seasonality=["monthly"])
days_obsoletes=180

/home/heylucasleao/tinyconformal/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
horizon = 12
train = df[:-horizon]
test = df[-horizon:]
def create_mlforecast_multiquantile():
     # Define 90% and 50% prediction interval quantile LightGBM models
     models = {
         "LGBM-lo-90": lgb.LGBMRegressor(objective="quantile", alpha=0.05, random_state=42),
         "LGBM-hi-90": lgb.LGBMRegressor(objective="quantile", alpha=0.95, random_state=42),
         "LGBM-lo-50": lgb.LGBMRegressor(objective="quantile", alpha=0.25, random_state=42),
         "LGBM-hi-50": lgb.LGBMRegressor(objective="quantile", alpha=0.75, random_state=42),
     }
     return MLForecast(
         models=models,
         freq="MS",
         lags=[1, 7],
     )
models = create_mlforecast_multiquantile()

In [35]:
cqr = ConformalQuantileTimeSeriesRegressor(
     learner=models,
     horizon=12,
     quantile_cols=[
         ("LGBM-lo-90", "LGBM-hi-90"),
         ("LGBM-lo-50", "LGBM-hi-50"),
     ],
     n_windows=10,
     alpha=0.10,
 )

In [36]:
cqr.fit(train, static_features=[])

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000022 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 95
[LightGBM] [Info] Number of data points in the train set: 113, number of used features: 4
[LightGBM] [Info] Start training from score 125.599998
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be

,learner,MLForecast(mo...num_threads=1)
,horizon,12
,quantile_cols,"[('LGBM-lo-90', ...), ('LGBM-lo-50', ...)]"
,n_windows,10
,alpha,0.1
,id_col,'unique_id'
,time_col,'ds'
,target_col,'y'


In [37]:
cqr.predict_interval(h=7, X_df=test)

,unique_id,ds,LGBM-lo-90,LGBM-hi-90,LGBM-lo-50,LGBM-hi-50,LGBM-lo-90-cqr,LGBM-hi-90-cqr,LGBM-lo-50-cqr,LGBM-hi-50-cqr
0,1,1960-01-01,318.219638,517.690094,366.396760,425.001579,315.220302,520.689430,342.397053,449.001287
1,1,1960-02-01,317.742386,517.690094,359.771495,425.665280,314.743050,520.689430,335.771788,449.664988
2,1,1960-03-01,349.328867,523.290530,351.918204,441.131212,306.329531,566.289867,287.918496,505.130920
3,1,1960-04-01,347.872791,542.627306,349.231564,439.608274,343.472454,547.027643,310.540907,478.298932
4,1,1960-05-01,350.880536,546.870603,350.215389,456.242684,348.880536,548.870603,318.371832,488.086240
5,1,1960-06-01,338.405857,546.858059,376.098383,485.476750,313.405857,571.858059,291.948455,569.626677
6,1,1960-07-01,343.390859,548.136656,425.510350,489.803914,274.695655,616.831860,305.372102,609.942163
